In [1]:
#%pip install transformers torch pandas scikit-learn -q

import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification, Trainer, TrainingArguments
from torch.utils.data import Dataset

In [2]:
import os
from sklearn.model_selection import train_test_split

full_train = pd.read_csv("train.csv")
# 80:20 train/val split from train.csv
tr_df, val_df = train_test_split(full_train, test_size=0.2, random_state=42, shuffle=True)
train = tr_df.reset_index(drop=True)
val = val_df.reset_index(drop=True)

# Dev file is used only for predictions.csv (not for training metrics)
dev_path = "dev.csv"
if os.path.exists(dev_path):
    dev = pd.read_csv(dev_path)
else:
    dev = pd.DataFrame(columns=["Input Sentence"])

print("Train split samples:", len(train))
print("Val split samples:", len(val))
print("Dev samples:", len(dev))


Train split samples: 440
Val split samples: 110
Dev samples: 100


In [3]:
def make_bio(input_text, output_text):
    input_tokens = input_text.strip().split()
    output_tokens = output_text.strip().split()
    labels = ["O"] * len(input_tokens)
    i = 0
    for group in output_tokens:
        parts = group.split("__")
        if len(parts) == 1:
            labels[i] = "O"
            i += 1
        else:
            for j, _ in enumerate(parts):
                labels[i] = "B" if j == 0 else "I"
                i += 1
    return input_tokens, labels

train_texts, train_labels = [], []
for i in range(len(train)):
    tokens, labels = make_bio(train.iloc[i]["Input Sentence"], train.iloc[i]["Output Sentence"])
    train_texts.append(tokens)
    train_labels.append(labels)

# Build validation split texts/labels from the 20% of train.csv
val_texts, val_labels = [], []
for i in range(len(val)):
    tokens, labels = make_bio(val.iloc[i]["Input Sentence"], val.iloc[i]["Output Sentence"])
    val_texts.append(tokens)
    val_labels.append(labels)

# Dev texts/labels are used only to create predictions.csv; labels may be absent
dev_texts, dev_labels = [], []
if "Output Sentence" in dev.columns:
    for i in range(len(dev)):
        tokens, labels = make_bio(dev.iloc[i]["Input Sentence"], dev.iloc[i]["Output Sentence"])
        dev_texts.append(tokens)
        dev_labels.append(labels)
else:
    print("Warning: 'Output Sentence' column not found in dev.csv - creating dummy 'O' labels.")
    for i in range(len(dev)):
        tokens = dev.iloc[i]["Input Sentence"].strip().split()
        dev_texts.append(tokens)
        dev_labels.append(["O"] * len(tokens))

print("Example tokens+labels (train sample):\n", list(zip(train_texts[0][:10], train_labels[0][:10])))


Example tokens+labels (train sample):
 [('खेल', 'O'), ('घास', 'B'), ('पर', 'I'), ('खेला', 'B'), ('जाता', 'I'), ('है', 'I'), (',', 'O'), ('और', 'O'), ('छेद', 'B'), ('के', 'I')]


In [12]:
model_name = "google/muril-base-cased" #"ai4bharat/indic-bert"  # or 

tokenizer = AutoTokenizer.from_pretrained(model_name)
label2id = {"B": 0, "I": 1, "O": 2}
id2label = {v: k for k, v in label2id.items()}

def encode_data(texts, labels):
    encodings = tokenizer(texts, is_split_into_words=True, truncation=True, padding=True)
    encoded_labels = []
    for i, label in enumerate(labels):
        word_ids = encodings.word_ids(batch_index=i)
        label_ids = []
        for word_id in word_ids:
            if word_id is None:
                label_ids.append(-100)
            else:
                label_ids.append(label2id[label[word_id]])
        encoded_labels.append(label_ids)
    encodings["labels"] = encoded_labels
    return encodings

train_enc = encode_data(train_texts, train_labels)
val_enc = encode_data(val_texts, val_labels)
dev_enc = encode_data(dev_texts, dev_labels)

class GroupDataset(Dataset):
    def __init__(self, encodings):
        self.encodings = encodings
    def __len__(self):
        return len(self.encodings["input_ids"])
    def __getitem__(self, idx):
        return {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}

train_dataset = GroupDataset(train_enc)
val_dataset = GroupDataset(val_enc)
dev_dataset = GroupDataset(dev_enc)


tokenizer_config.json:   0%|          | 0.00/206 [00:00<?, ?B/s]

c:\Users\MANAV DHAMECHA\miniconda3\envs\gec\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\MANAV DHAMECHA\.cache\huggingface\hub\models--google--muril-base-cased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config.json:   0%|          | 0.00/411 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/113 [00:00<?, ?B/s]

In [13]:
model = AutoModelForTokenClassification.from_pretrained(model_name, num_labels=3, id2label=id2label, label2id=label2id)

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


pytorch_model.bin:   0%|          | 0.00/953M [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/953M [00:00<?, ?B/s]

Some weights of BertForTokenClassification were not initialized from the model checkpoint at google/muril-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [14]:
# Create TrainingArguments in a backwards-compatible way (some transformer versions do not accept evaluation_strategy)
try:
    args = TrainingArguments(
        output_dir="./results",
        eval_strategy="epoch",
        logging_dir="./logs",
        learning_rate=2e-5,
        per_device_train_batch_size=8,
        num_train_epochs=4,
        weight_decay=0.01,
        save_total_limit=1,
    )
except TypeError as e:
    print("TrainingArguments does not accept 'evaluation_strategy' in this transformers version. Falling back to a compatible signature.")
    args = TrainingArguments(
        output_dir="./results",
        logging_dir="./logs",
        learning_rate=2e-5,
        per_device_train_batch_size=8,
        num_train_epochs=4,
        weight_decay=0.01,
        save_total_limit=1,
    )
    # If possible, set the attribute so Trainer can still pick it up
    try:
        args.evaluation_strategy = "epoch"
    except Exception:
        pass

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
)

trainer.train()

C:\Users\MANAV DHAMECHA\AppData\Local\Temp\ipykernel_129732\84216367.py:30: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss
1,No log,1.018715
2,No log,0.935461
3,No log,0.902651
4,No log,0.896156


TrainOutput(global_step=220, training_loss=0.9720003995028409, metrics={'train_runtime': 134.9214, 'train_samples_per_second': 13.045, 'train_steps_per_second': 1.631, 'total_flos': 208386047485440.0, 'train_loss': 0.9720003995028409, 'epoch': 4.0})

In [15]:
# Run predictions on dev dataset and reconstruct output sentences (for predictions.csv only)
import numpy as np
# Get raw predictions (logits) and convert to label ids
pred_output = trainer.predict(dev_dataset)
preds = np.argmax(pred_output.predictions, axis=-1)

# Re-tokenize dev_texts to get word_ids mapping (must match encoding used for predictions)
tokenized_dev = tokenizer(dev_texts, is_split_into_words=True, truncation=True, padding=True)
outputs_sentences = []
for i in range(len(dev_texts)):
    word_ids = tokenized_dev.word_ids(batch_index=i)
    pred_labels_for_tokens = preds[i]
    # Build a list of predicted labels aligned to words (skip special tokens where word_id is None)
    word_labels = []
    last_word_idx = None
    for token_idx, word_idx in enumerate(word_ids):
        if word_idx is None:
            continue
        if word_idx != last_word_idx:
            label_id = int(pred_labels_for_tokens[token_idx])
            if label_id == -100:
                word_labels.append('O')
            else:
                word_labels.append(id2label.get(label_id, 'O'))
            last_word_idx = word_idx
        else:
            continue
    # Reconstruct output sentence
    words = dev_texts[i]
    out_tokens = []
    w = 0
    while w < len(words):
        lab = word_labels[w] if w < len(word_labels) else 'O'
        if lab == 'O':
            out_tokens.append(words[w])
            w += 1
        else:
            if lab == 'B':
                group = [words[w]]
                w += 1
                while w < len(words) and (w < len(word_labels) and word_labels[w] == 'I'):
                    group.append(words[w])
                    w += 1
                out_tokens.append('__'.join(group))
            else:
                out_tokens.append(words[w])
                w += 1
    outputs_sentences.append(' '.join(out_tokens))

# Save predictions to CSV (dev only)
out_df = pd.DataFrame({
    "Input Sentence": dev.get("Input Sentence", pd.Series([""]*len(outputs_sentences))),
    "Output Sentence": outputs_sentences
})
out_df.to_csv("predictions.csv", index=False)
print("✅ Saved predictions.csv successfully!")


✅ Saved predictions.csv successfully!


In [16]:
# ========================================
# 8. Evaluation (Exact Match Accuracy on 20% of train.csv)
# ========================================

# Predict on validation split (20% of train.csv) and compute exact-match
val_pred_output = trainer.predict(val_dataset)
val_preds = np.argmax(val_pred_output.predictions, axis=-1)

# Re-tokenize val_texts to get word_ids mapping
tokenized_val = tokenizer(val_texts, is_split_into_words=True, truncation=True, padding=True)
val_outputs_sentences = []
for i in range(len(val_texts)):
    word_ids = tokenized_val.word_ids(batch_index=i)
    pred_labels_for_tokens = val_preds[i]
    word_labels = []
    last_word_idx = None
    for token_idx, word_idx in enumerate(word_ids):
        if word_idx is None:
            continue
        if word_idx != last_word_idx:
            label_id = int(pred_labels_for_tokens[token_idx])
            if label_id == -100:
                word_labels.append('O')
            else:
                word_labels.append(id2label.get(label_id, 'O'))
            last_word_idx = word_idx
        else:
            continue
    words = val_texts[i]
    out_tokens = []
    w = 0
    while w < len(words):
        lab = word_labels[w] if w < len(word_labels) else 'O'
        if lab == 'O':
            out_tokens.append(words[w])
            w += 1
        else:
            if lab == 'B':
                group = [words[w]]
                w += 1
                while w < len(words) and (w < len(word_labels) and word_labels[w] == 'I'):
                    group.append(words[w])
                    w += 1
                out_tokens.append('__'.join(group))
            else:
                out_tokens.append(words[w])
                w += 1
    val_outputs_sentences.append(' '.join(out_tokens))

# Compute exact match on validation split
gold = val["Output Sentence"].astype(str).tolist()
preds_em = [s for s in val_outputs_sentences]
total = len(gold)
matches = sum(1 for g, p in zip(gold, preds_em) if g.strip() == p.strip())
accuracy = (matches / total * 100.0) if total else 0.0

# Compute token F1 (macro and micro) using whitespace token overlap
def token_sets(text):
    return set(str(text).lower().split())

macro_f1_list = []
tp_sum = 0
pred_sum = 0
gold_sum = 0
for g, p in zip(gold, preds_em):
    gt = token_sets(g)
    pt = token_sets(p)
    inter = gt & pt
    prec = len(inter) / len(pt) if len(pt) else 0.0
    rec = len(inter) / len(gt) if len(gt) else 0.0
    f1 = (2 * prec * rec / (prec + rec)) if (prec + rec) else 0.0
    macro_f1_list.append(f1)
    tp_sum += len(inter)
    pred_sum += len(pt)
    gold_sum += len(gt)
macro_f1 = float((sum(macro_f1_list) / len(macro_f1_list)) * 100.0) if macro_f1_list else 0.0
micro_prec = tp_sum / pred_sum if pred_sum else 0.0
micro_rec = tp_sum / gold_sum if gold_sum else 0.0
micro_f1 = float((2 * micro_prec * micro_rec / (micro_prec + micro_rec)) * 100.0) if (micro_prec + micro_rec) else 0.0

print("✅ Validation (20% of train.csv) Results:")
print(f"Total Samples: {total}")
print(f"Exact Matches: {matches}")
print(f"Exact Match Accuracy: {accuracy:.2f}%")
print(f"Token F1 (macro, whitespace overlap): {macro_f1:.2f}%")
print(f"Token F1 (micro, whitespace overlap): {micro_f1:.2f}%")

# Optionally, view up to 5 mismatches
shown = 0
for i, (g, p) in enumerate(zip(gold, preds_em)):
    if g.strip() != p.strip():
        print(f"Index: {i}")
        print(f"Input: {val.iloc[i]['Input Sentence']}")
        print(f"Gold:  {g}")
        print(f"Pred:  {p}")
        shown += 1
        if shown >= 5:
            break


✅ Validation (20% of train.csv) Results:
Total Samples: 110
Exact Matches: 46
Exact Match Accuracy: 41.82%
Token F1 (macro, whitespace overlap): 91.56%
Token F1 (micro, whitespace overlap): 91.66%
Index: 0
Input: मास्सा कम से कम बाकी के 2009 सीज़न के लिए बाहर हो सकते हैं ।
Gold:  मास्सा कम से कम बाकी__के 2009 सीज़न__के__लिए बाहर__हो__सकते__हैं ।
Pred:  मास्सा कम से कम बाकी__के 2009 सीज़न__के__लिए बाहर हो__सकते__हैं ।
Index: 2
Input: अठारह छेद एक नियमित दौर के दौरान खेले जाते हैं , खिलाड़ी आमतौर पर मैदान के पहले छेद से शुरू करते हैं और अठारहवें पर समाप्त करते हैं .
Gold:  अठारह छेद एक नियमित दौर__के दौरान खेले__जाते__हैं , खिलाड़ी आमतौर पर मैदान__के पहले छेद__से शुरू__करते__हैं और अठारहवें__पर समाप्त__करते__हैं .
Pred:  अठारह छेद एक नियमित दौर__के दौरान खेले__जाते__हैं , खिलाड़ी आमतौर पर मैदान__के पहले छेद__से शुरू__करते__हैं और अठारहवें पर समाप्त__करते__हैं .
Index: 3
Input: कैप्सूल एक मिनट में सैन फ़्रांसिस्को से लॉस एंजिलस तेज़ी से जाने के लिए लगभग 12.8 किमी या 8 मील प्रति सेकंड की यात्